# Climate Statistics Dashboard

Interactive explorer for ERA5/IFS climate metrics across Illinois counties and ZIP codes.

**Run with Voila:** `voila dashboard.ipynb`  
**Or use as a regular notebook in Jupyter.**

In [ ]:
import os
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

from climstat.data_visualization.dashboard_data import (
    scan_output_dir,
    get_available_metrics,
    get_date_ranges,
    load_metric_data,
    get_daily_columns,
    get_averages_columns,
    get_regions,
)
from climstat.data_visualization.visualization import (
    plot_county_timeseries,
    plot_county_map,
    plot_zipcode_map,
)
from climstat.data_process.shapefiles import load_illinois_counties, load_illinois_zctas

%matplotlib inline

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "data")
OUTPUT_DIR = os.path.join(DATA_DIR, "output")
SHAPEFILE_PATH = os.path.join(DATA_DIR, "shapefiles", "derived", "tl_2025_il_counties_land_only.shp")
ZCTA_SHAPEFILE_PATH = os.path.join(DATA_DIR, "shapefiles", "zipcodes", "tl_2025_us_zcta520.shp")


# ── Scan available data ────────────────────────────────────────────────
catalog = scan_output_dir(OUTPUT_DIR)
metrics = get_available_metrics(catalog)
date_ranges = get_date_ranges(catalog)

# ── Pre-load shapefiles (once) ─────────────────────────────────────────
counties_gdf = load_illinois_counties(SHAPEFILE_PATH)
zctas_gdf = load_illinois_zctas(ZCTA_SHAPEFILE_PATH, SHAPEFILE_PATH)

print(f"Found {len(metrics)} metrics: {metrics}")
print(f"Date ranges: {date_ranges}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Global controls
# ══════════════════════════════════════════════════════════════════════

w_metric = widgets.Dropdown(
    options=metrics,
    value=metrics[0] if metrics else None,
    description="Metric:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="300px"),
)

w_region_type = widgets.ToggleButtons(
    options=[("Counties", "counties"), ("ZIP Codes", "zipcodes")],
    value="counties",
    description="Region:",
    style={"description_width": "80px", "button_width": "100px"},
)

global_controls = widgets.HBox([w_metric, w_region_type])

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Helper: load data for current metric + region type
# ══════════════════════════════════════════════════════════════════════

def _current_data():
    """Load ERA5 + IFS data for the currently selected metric/region."""
    metric = w_metric.value
    region = w_region_type.value
    era5 = load_metric_data(catalog, metric, "ERA5", region)
    ifs = load_metric_data(catalog, metric, "IFS", region)
    return era5, ifs


def _region_col():
    return "NAMELSAD" if w_region_type.value == "counties" else "ZCTA5CE20"

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Tab 1: Time Series Explorer
# ══════════════════════════════════════════════════════════════════════

w_ts_region = widgets.Dropdown(
    description="Region:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="300px"),
)

w_ts_stat = widgets.Dropdown(
    description="Statistic:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="300px"),
)

w_ts_recent = widgets.Checkbox(
    value=False,
    description="Recent 60 days",
    layout=widgets.Layout(width="200px"),
)

w_ts_button = widgets.Button(
    description="Plot",
    button_style="primary",
    layout=widgets.Layout(width="80px"),
)

ts_output = widgets.Output()

ts_controls = widgets.HBox([w_ts_region, w_ts_stat, w_ts_recent, w_ts_button])
ts_tab = widgets.VBox([ts_controls, ts_output])


def _update_ts_dropdowns(*_):
    """Refresh region and stat dropdowns when metric or region type changes."""
    era5, _ = _current_data()
    daily = era5["daily"]
    # Update region choices
    regions = get_regions(daily, w_region_type.value)
    w_ts_region.options = regions
    if regions:
        w_ts_region.value = regions[0]
    # Update stat choices
    cols = get_daily_columns(daily)
    w_ts_stat.options = cols
    if cols:
        w_ts_stat.value = cols[0]


def _plot_timeseries(_=None):
    """Draw the time series plot."""
    ts_output.clear_output(wait=True)
    with ts_output:
        era5, ifs = _current_data()
        daily = era5["daily"]
        ifs_daily = ifs["daily"]
        if daily is None:
            print("No data available for this metric.")
            return
        stat = w_ts_stat.value
        if stat not in daily.columns:
            print(f"Column '{stat}' not found.")
            return

        region_val = w_ts_region.value
        region_col = _region_col()

        date_range = None
        marker = None
        if w_ts_recent.value:
            import pandas as pd
            start = (pd.Timestamp.today() - pd.Timedelta(days=60)).strftime("%Y-%m-%d")
            end = (pd.Timestamp.today() + pd.Timedelta(days=14)).strftime("%Y-%m-%d")
            date_range = (start, end)
            marker = "o"

        fig, ax = plt.subplots(figsize=(12, 4))
        plot_county_timeseries(
            daily,
            county=region_val,
            daily_summary=stat,
            region_col=region_col,
            date_range=date_range,
            title=f"{w_metric.value}: {stat} — {region_val}",
            ax=ax,
            df_forecast=ifs_daily,
            marker=marker,
        )
        plt.show()
        plt.close(fig)


w_metric.observe(_update_ts_dropdowns, names="value")
w_region_type.observe(_update_ts_dropdowns, names="value")
w_ts_button.on_click(_plot_timeseries)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Tab 2: Spatial Map
# ══════════════════════════════════════════════════════════════════════

w_map_stat = widgets.Dropdown(
    description="Statistic:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="350px"),
)

w_map_cmap = widgets.Dropdown(
    options=["YlOrRd", "Blues", "coolwarm", "RdYlGn_r", "viridis", "plasma"],
    value="YlOrRd",
    description="Colormap:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="200px"),
)

w_map_button = widgets.Button(
    description="Plot",
    button_style="primary",
    layout=widgets.Layout(width="80px"),
)


map_output = widgets.Output()

map_controls = widgets.HBox([w_map_stat, w_map_cmap, w_map_button])
map_tab = widgets.VBox([map_controls, map_output])


def _update_map_dropdowns(*_):
    """Refresh map stat dropdown when metric or region type changes."""
    era5, _ = _current_data()
    avgs = era5["averages"]
    cols = get_averages_columns(avgs)
    w_map_stat.options = cols
    if cols:
        w_map_stat.value = cols[0]


def _plot_map(_=None):
    """Draw the choropleth map."""
    map_output.clear_output(wait=True)
    with map_output:
        era5, _ = _current_data()
        avgs = era5["averages"]
        if avgs is None:
            print("No averages data available for this metric.")
            return
        stat = w_map_stat.value
        if stat not in avgs.columns:
            print(f"Column '{stat}' not found.")
            return

        fig, ax = plt.subplots(figsize=(8, 10))
        if w_region_type.value == "counties":
            plot_county_map(
                avgs,
                average_summary=stat,
                title=f"{w_metric.value}: {stat} — Counties",
                cmap=w_map_cmap.value,
                ax=ax,
                gdf=counties_gdf,
            )
        else:
            plot_zipcode_map(
                avgs,
                average_summary=stat,
                title=f"{w_metric.value}: {stat} — ZIP Codes",
                cmap=w_map_cmap.value,
                ax=ax,
                gdf=zctas_gdf,
            )
        plt.show()
        plt.close(fig)


w_metric.observe(_update_map_dropdowns, names="value")
w_region_type.observe(_update_map_dropdowns, names="value")
w_map_button.on_click(_plot_map)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Assemble tabs and display
# ══════════════════════════════════════════════════════════════════════

tabs = widgets.Tab(children=[ts_tab, map_tab])
tabs.set_title(0, "Time Series")
tabs.set_title(1, "Spatial Map")

# Initialize dropdowns with first metric's data
_update_ts_dropdowns()
_update_map_dropdowns()

display(global_controls, tabs)